# Block diffusion language model (`variant: block_diffusion`)

<img src="../../assets/explainers/block_diffusion.gif" width="100%">

**Training**: two streams, a clean copy and a masked copy with one $t$ per block; the sweep shows that noised block $b$ reads only the clean blocks before it and itself (cyan links, and the matching row of the block-causal mask). **Generation**: blocks left to right (orange outline), a short masked diffusion $t: 1 \to 0$ inside each; later blocks do not exist yet.

## Intuition

Autoregressive models can stop anywhere and reuse their past; diffusion models fill many tokens per step.
Block diffusion (Arriola et al. 2025) does both: autoregressive **across** blocks of $B$ tokens, masked
diffusion **within** the current block, conditioned on all previous clean blocks.

## Math

$$p_\theta(x) = \prod_{b} p_\theta\big(x^{(b)} \mid x^{(<b)}\big),\qquad
  \mathcal{L} = \sum_b \mathbb{E}_{t_b}\Big[\frac{1}{t_b}\sum_{i \in b,\ \text{masked}} -\log p_\theta(x^i \mid x_{t_b}^{(b)}, x^{(<b)})\Big].$$

**Two streams, one forward.** The backbone input is $[x ; x_t]$ (length $2N$). The `block_causal` mask
lets clean token $i$ see clean blocks $\le$ its own, and noised block $b$ see clean blocks $< b$ plus
itself. All blocks, each at its own noise level, train in one pass.

**Sampling.** For block $b$: start all-masked at $t=1$, run `sampling_steps` reveal steps with probability
$(t-s)/t$, greedy-fill leftovers, freeze the block, move on.

## In the code

| File | What |
|---|---|
| `models_block_diffusion/model.py` | `BlockDiffusionLM` (two-stream forward), `block_diffusion_loss`, `sample_block_diffusion` |
| HaleBlocks `build_attn_mask("block_causal", n, block_size=B)` | the $2N \times 2N$ mask |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Block-causal attention, blocks of 8, time conditioning. The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("block_diffusion")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## The block-causal mask over [clean ; noised]

In [ ]:
from hale_core.nn.layers.attention import build_attn_mask

m = build_attn_mask("block_causal", 16, "cpu", block_size=4)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(~m, cmap="Blues"); ax.axhline(15.5, color="#F4A261"); ax.axvline(15.5, color="#F4A261")
ax.set(title="16 tokens, blocks of 4: blue = allowed", xlabel="key", ylabel="query")
ax.text(3, 7, "clean→clean\n(block-causal)", fontsize=8, color="white")
ax.text(18, 27, "noised→\nown block", fontsize=8)
plt.show()

## A training example: one time per block

In [ ]:
x = run.ids
torch.manual_seed(1)
t_block = torch.rand(1, 6) * (1 - 1e-3) + 1e-3
t_noised = t_block.repeat_interleave(8, dim=1)
masked = torch.rand(1, 48) < t_noised
for b in range(6):
    seg = "".join("_" if masked[0, i] else run.text[i] for i in range(b * 8, b * 8 + 8))
    print(f"block {b}: t={float(t_block[0, b]):.2f}  {seg!r}")
with torch.no_grad():
    print("loss:", float(get_loss("block_diffusion")(model, get_collate("block_diffusion")(x, tok))))

In [ ]:
def validity(steps_list, seeds=24, temperature=1.0):
    """Share of samples that are exactly one of the training sentences, per sampling-step budget."""
    prompt = run.ids[:, :8]
    sampler = get_sampler(cfg.variant)
    out = {}
    for steps in steps_list:
        ok = 0
        for seed in range(seeds):
            torch.manual_seed(seed)
            ids = sampler(model, tok, prompt, max_new_tokens=40, temperature=temperature, sampling_steps=steps)
            ok += tok.decode(ids[0]) in CORPUS
        out[steps] = ok / seeds
    return out

In [ ]:
scores = validity([1, 2, 4, 8])
print("valid share vs steps per block:", scores)

In [ ]:
trace = explainer.trace(run, seed=0)
assert torch.equal(trace.ids, explainer.reference(run, seed=0))
for s in trace.steps:
    canvas = "".join("_" if v == tok.mask_token_id else tok.token(v) for v in s.x)
    tag = "fill" if s.final_fill else f"t {s.t_from:.2f}→{s.t_to:.2f}"
    print(f"block {s.block} {tag:13} +{len(s.revealed)}  {canvas}")

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/block_diffusion.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "block_diffusion.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/block_diffusion.gif" width="100%">'))

## Takeaways

* Variable-length generation and reusable clean context (KV-cacheable) like AR; parallel fills inside a
  block like diffusion. Block size interpolates between the two (B=1 is autoregressive).
* Mixing errors are confined to a block - and still possible (look for "rodel" in the GIF).

**Exercise.** Train with `block_size: 4` and `16` (`train_toy("block_diffusion", CORPUS, model={...})`)
and compare `validity` at 2 steps per block.